In [39]:
# Importar as bibliotecas
import pandas as pd 
import numpy as np


ENDERECO_DADOS = 'roubos_flag.csv'


# Lendo o arquivo CSV
try:
    df_roubos = pd.read_csv(ENDERECO_DADOS, sep=';', encoding='utf-8-sig')

    print(df_roubos.head())

except Exception as e:
    print("Erro ao ler csv: ", e)

   cisp   regiao           munic  roubo_veiculo  flag
0    17  Capital  Rio de Janeiro           7771  mais
1    18  Capital  Rio de Janeiro           7393  mais
2    20  Capital  Rio de Janeiro           9201  mais
3    21  Capital  Rio de Janeiro          19884  mais
4    22  Capital  Rio de Janeiro          15236  mais


In [30]:
# Jogar o dataframe como uma tabela no Banco de dados
from sqlalchemy import create_engine, text


try:
    host = 'localhost'
    user = 'root'
    password = ''
    database = 'bd_ocorrencias_maiores_menores'

    engine = create_engine(f'mysql+pymysql://{user}:{password}@{host}/{database}')

    tabela = 'tb_roubo_veiculos'

    # 'replace': cria a tabela ou substitui a existente.
    # 'fail': cria a tabela ou gera erro se ela já existir.
    # 'append': cria a tabela ou adiciona dados à existente.
    df_roubos.to_sql(tabela, con=engine, if_exists='replace', index=False)

    print("Dados armazenados com sucesso!")

except Exception as e:
    print("Erro ao exportar os dados: ", e)

Dados armazenados com sucesso!


CRUD READ OPÇÃO 01 - Uso na progaramação em Geral

In [31]:
# READ
# Estrutura with para abrir e fechar o arquivo
with engine.connect() as conexao:
    comando_sql = text("SELECT * FROM tb_roubo_veiculos")
    resultado = conexao.execute(comando_sql)
    for linha in resultado:
        print(linha)

(17, 'Capital', 'Rio de Janeiro', 7771, 'mais')
(18, 'Capital', 'Rio de Janeiro', 7393, 'mais')
(20, 'Capital', 'Rio de Janeiro', 9201, 'mais')
(21, 'Capital', 'Rio de Janeiro', 19884, 'mais')
(22, 'Capital', 'Rio de Janeiro', 15236, 'mais')
(23, 'Capital', 'Rio de Janeiro', 10833, 'mais')
(24, 'Capital', 'Rio de Janeiro', 12311, 'mais')
(25, 'Capital', 'Rio de Janeiro', 11140, 'mais')
(26, 'Capital', 'Rio de Janeiro', 7920, 'mais')
(27, 'Capital', 'Rio de Janeiro', 28171, 'mais')
(28, 'Capital', 'Rio de Janeiro', 11120, 'mais')
(29, 'Capital', 'Rio de Janeiro', 17260, 'mais')
(30, 'Capital', 'Rio de Janeiro', 12987, 'mais')
(31, 'Capital', 'Rio de Janeiro', 15283, 'mais')
(32, 'Capital', 'Rio de Janeiro', 8854, 'mais')
(33, 'Capital', 'Rio de Janeiro', 18285, 'mais')
(34, 'Capital', 'Rio de Janeiro', 27785, 'mais')
(35, 'Capital', 'Rio de Janeiro', 19640, 'mais')
(38, 'Capital', 'Rio de Janeiro', 20218, 'mais')
(39, 'Capital', 'Rio de Janeiro', 30059, 'mais')
(40, 'Capital', 'Rio de J

CRUD READ OPÇÃO 02 - Uso do READ no Pandas

In [32]:
# READ - Consultar dados da tabela
comando_sql = 'SELECT * FROM tb_roubo_veiculos'
df = pd.read_sql(comando_sql, engine)

# ou também pode o nome da tabela no banco de dados
# df = pd.read_sql('tb_roubo_veiculos', engine)  

print(df)

    ï»¿cisp    regiao                 munic  roubo_veiculo   flag
0        17   Capital        Rio de Janeiro           7771   mais
1        18   Capital        Rio de Janeiro           7393   mais
2        20   Capital        Rio de Janeiro           9201   mais
3        21   Capital        Rio de Janeiro          19884   mais
4        22   Capital        Rio de Janeiro          15236   mais
..      ...       ...                   ...            ...    ...
69      156  Interior  Santa Maria Madalena             47  menos
70      157  Interior     Trajano de Morais             21  menos
71      158  Interior            Bom Jardim             21  menos
72      167   Capital                Paraty              2  menos
73      168  Interior             Rio Claro             66  menos

[74 rows x 5 columns]


CREATE

In [35]:
# CREATE
try: 
    with engine.connect() as conexao:
        comando_sql = text(
            """
                INSERT INTO tb_roubo_veiculos (cisp, regiao, munic, roubo_veiculo, flag)
                VALUES (171, 'Grande Niterói', 'Sao Gonçalo', 500, 'mais')
            """
        )
        
        conexao.execute(comando_sql)

        conexao.commit()

except Exception as e:
    print(e)

CREATE PANDAS

In [36]:
# Criar um DataFrame com o novo registro
novo_registro = pd.DataFrame({
    'cisp': [171],
    'regiao': ['Grande Niterói'],
    'munic': ['São Gonçalo'],
    'roubo_veiculo': [500],
    'flag': ['mais']
})

# Inserir o registro na tabela existente
novo_registro.to_sql(
    'tb_roubo_veiculos',
    engine,
    if_exists='append',
    index=False
)

1

In [37]:
# UPDATE
# Onde a cisp é 171, passa a ser 76,
# e atualizando para São Gonçalo, flag 'menos', roubo 20 etc
with engine.connect() as conexao:

    regiao = 'Grande Niterói'
    munic = 'Niterói'
    roubo_veiculo = 20
    flag = 'menos'
    cisp_nova = 76
    cisp_atual = 171

    comando_sql = text(
        """
            UPDATE tb_roubo_veiculos
            SET regiao = :regiao,
                cisp = :cisp_nova,
                munic = :munic,
                roubo_veiculo = :roubo_veiculo,
                flag = :flag
            WHERE cisp = :cisp_atual
        """
    )

    conexao.execute(comando_sql, {
        'regiao': regiao,
        'cisp_nova': cisp_nova,
        'munic': munic,
        'roubo_veiculo': roubo_veiculo,
        'flag': flag,
        'cisp_atual': cisp_atual
        }
    )
    
    conexao.commit()

In [38]:
# DELETE
with engine.connect() as conexao:

    comando_sql = text(
        """
            DELETE FROM tb_roubo_veiculos
            WHERE cisp = :cisp
        """
    )

    conexao.execute(comando_sql, {
        'cisp': 69
        }
    )

    conexao.commit()

    # Sugestão:
    # Com engine.connect(), normalmente é necessário o .commit()
    # Usar engine.begin() no lugar de engine.connect(), 
    # para que o commit seja automático: